In [1]:
!pip install llama-cpp-python
!pip install langchain langchain-community pandas langchain_qdrant langchain_huggingface



In [2]:
import pandas as pd
from llama_cpp import Llama
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings


In [3]:
excel_path = "data/lsst_forum_responses_5.xlsx"
df = pd.read_excel(excel_path)

questions = df['question'].dropna().tolist()

In [4]:
# GGUF model paths and display names
model_configs = [
    {"path": "gguf_models/OLMo-2-0425-1B-Instruct-Q4_K_M.gguf", "name": "OLMo-2-0425-1B-Instruct-Q4_K_M"},
    # Add more like this
    # {"path": "other_model.gguf", "name": "Other-Model"},
]

In [5]:
# Helper to load GGUF models
def load_llama_model(model_path):
    return Llama(
        model_path=model_path,
        n_ctx=2048,
        n_gpu_layers=0  # Use CPU; change if using GPU
    )


In [6]:
qdrant_path = "resources/rubin_qdrant"
collection = "rubin_telescope"

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
client = QdrantClient(path=qdrant_path)
vector_db = Qdrant(client=client, collection_name=collection, embeddings=embedding)
retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 2})

def get_context(question):
    docs = retriever.get_relevant_documents(question)
    return "\n\n".join([doc.page_content for doc in docs])

/Users/baisakhisarkar/Downloads/OPT_UW_Temp/OLMO1_OLMO2_Data/olmo-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/6n/w6q7td3d1r72qclcnk1t5skc0000gn/T/ipykernel_32309/2098024014.py:6: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use :class:`~QdrantVectorStore` instead.
  vector_db = Qdrant(client=client, collection_name=collection, embeddings=embedding)


In [7]:
def generate_from_gguf_models_with_rag(questions, model_configs, max_tokens=200, temperature=0.7):
    results = {"question": questions}
    
    for model in model_configs:
        print(f"\n🔍 Loading model: {model['name']}")
        llm = load_llama_model(model['path'])

        outputs = []
        for q in questions:
            context = get_context(q)
            prompt = f"""You are an astrophysics expert. Use the following context to answer the question.

                        Context:
                        {context}

                        Question:
                        {q}
                        """
            response = llm.create_completion(
                prompt=prompt,
                max_tokens=max_tokens,
                temperature=temperature
            )
            answer = response["choices"][0]["text"].strip()
            outputs.append(answer)

        # Add RAG answers under a new column
        col_name = model["name"] + "_RAG"
        results[col_name] = outputs

    return pd.DataFrame(results)

In [8]:
final_df = generate_from_gguf_models_with_rag(questions, model_configs)
final_df.to_csv("gguf_model_responses_RAG.csv", index=False)
final_df.head()

llama_model_load_from_file_impl: using device Metal (Apple M2 Pro) - 9866 MiB free
llama_model_loader: loaded meta data with 42 key-value pairs and 179 tensors from gguf_models/OLMo-2-0425-1B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = olmo2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = OLMo 2 0425 1B Instruct
llama_model_loader: - kv   3:                             general.author str              = AllenAI
llama_model_loader: - kv   4:                       general.organization str              = AllenAI
llama_model_loader: - kv   5:                           general.finetune str              = Instruct
llama_model_loader: - kv   6:                    


🔍 Loading model: OLMo-2-0425-1B-Instruct-Q4_K_M


load_tensors: offloading 0 repeating layers to GPU
load_tensors: offloaded 0/17 layers to GPU
load_tensors:   CPU_Mapped model buffer size =   888.79 MiB
.....................................................................
llama_context: constructing llama_context
llama_context: n_seq_max     = 1
llama_context: n_ctx         = 2048
llama_context: n_ctx_per_seq = 2048
llama_context: n_batch       = 512
llama_context: n_ubatch      = 512
llama_context: causal_attn   = 1
llama_context: flash_attn    = 0
llama_context: freq_base     = 500000.0
llama_context: freq_scale    = 1
llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
ggml_metal_init: allocating
ggml_metal_init: found device: Apple M2 Pro
ggml_metal_init: picking default device: Apple M2 Pro
ggml_metal_load_library: using embedded metal library
ggml_metal_init: GPU name:   Apple M2 Pro
ggml_metal_init: GPU family: MTLGPUFamilyApple8  (1008)
ggml_metal_init: GPU family: M

,question,OLMo-2-0425-1B-Instruct-Q4_K_M_RAG
0,"Hi, \nI’m following this tutorial: The LSST S...",**[Note: The query is asking for a reasoning b...
1,I have the following C++ class : \n class CcdI...,This is the C++ code\n```c++\nclass CcdImageLi...
2,Question on how forced photometry will be run ...,Question 1:\n Q: When w...
3,"Hi there, \n Is there some way I find out what...",Answer:\nThe Butler repository management can ...
4,I’m having trouble building FFTW with texinfo ...,Any suggestions on how to get this working wou...
